# London August LME Diagnostics

Diagnostic plots for models M1-M14 fitted by `R/efus_mm_london_1month.R`.

Two model families:
- **Static (M1-M7)**: LME with i.i.d. residuals
- **AR(1) (M8-M14)**: LME with AR(1) residual correlation

Time-of-day treatments include none, sin/cos harmonics, and hour-of-day dummies. M4-M7 (static) and M11-M14 (AR(1)) additionally include binary and categorical building characteristics.

**Normalised residuals**: ACF, QQ, and residual-vs-fitted plots use `nlme` normalised residuals, i.e. residuals transformed by the fitted within-group residual-correlation structure and scaled by the model residual standard deviation. For static models these are essentially standardised conditional response residuals. For AR(1) models they are approximately whitened residuals; remaining autocorrelation indicates the AR(1) structure has not fully captured the serial dependence.

**OSA**: One-step-ahead predictions incorporate the lagged AR state: OSA_t = mean_t + rho*r_{t-1}, where r_{t-1} is the previous response residual.

In [ ]:
# Formatting of plots

import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
import os 

plt.style.use(['science', 'nature','bright'])
mpl.rcParams['savefig.format'] = 'svg'
os.makedirs('plots/efus2017/london_livingroom_1month', exist_ok=True)


In [ ]:
import os
# Paths below are relative to the project root (this notebook's folder);
# Jupyter starts the kernel there, so no chdir is needed.
assert os.path.isdir('diagnostics'), \
    'diagnostics/ not found: start Jupyter from the project root and run the R scripts first'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.patches import Patch

%matplotlib inline
import scienceplots  # noqa: F401
plt.style.use(['science', 'nature', 'bright'])
plt.rcParams.update({'figure.dpi': 120})
plt.rcParams['text.usetex'] = True

# -- Model metadata -----------------------------------------------------------
MODELS = ["M1", "M2", "M3", "M4", "M5", "M6", "M7",
          "M8", "M9", "M10", "M11", "M12", "M13", "M14"]
FAMILY = {
    "M1": "Static",  "M2": "Static",  "M3": "Static",
    "M4": "Static",  "M5": "Static",  "M6": "Static",  "M7": "Static",
    "M8": "AR(1)",   "M9": "AR(1)",   "M10": "AR(1)",
    "M11": "AR(1)",  "M12": "AR(1)",  "M13": "AR(1)",  "M14": "AR(1)",
}
TODS = {
    "M1": "none",          "M2": "sin/cos",        "M3": "hour",
    "M4": "hour+bin",      "M5": "hour+bin+cat",
    "M6": "sin/cos+bin",   "M7": "sin/cos+bin+cat",
    "M8": "none",          "M9": "sin/cos",        "M10": "hour",
    "M11": "hour+bin",     "M12": "hour+bin+cat",
    "M13": "sin/cos+bin",  "M14": "sin/cos+bin+cat",
}
FAMILY_COLOUR = {"Static": "#2E67D0", "AR(1)": "#27C05A"}
NROWS, NCOLS = 4, 4

# -- Defaults; overwritten by CSV exports from R/efus_mm_london_1month.R -------
AIC_VALUES = {m: float("nan") for m in MODELS}
AR_PARAMS  = {m: (None, None) for m in MODELS}
SIG_EPS    = {m: float("nan") for m in MODELS}
SIG_U0     = {m: float("nan") for m in MODELS}

_aic_path = "diagnostics/london_august_aic.csv"
_vc_path  = "diagnostics/london_august_varcomp.csv"
if os.path.exists(_aic_path):
    _aic = pd.read_csv(_aic_path)
    AIC_VALUES.update(dict(zip(_aic["model"], _aic["aic"])))
if os.path.exists(_vc_path):
    _vc = pd.read_csv(_vc_path).set_index("model")
    for m in MODELS:
        if m in _vc.index:
            rho = None if pd.isna(_vc.loc[m, "rho"]) else float(_vc.loc[m, "rho"])
            phi = None if pd.isna(_vc.loc[m, "phi"]) else float(_vc.loc[m, "phi"])
            AR_PARAMS[m] = (rho, phi)
            SIG_EPS[m]   = float(_vc.loc[m, "sig_eps"])
            SIG_U0[m]    = float(_vc.loc[m, "sig_u0"])

# -- % variance accounted for vs family-specific null model -------------------
NULL_FOR = {"Static": "M0_static", "AR(1)": "M0_ar1"}
PVAR = {}
if os.path.exists(_vc_path):
    for m in MODELS:
        if m in _vc.index:
            null_key = NULL_FOR[FAMILY[m]]
            if null_key in _vc.index:
                null_var  = float(_vc.loc[null_key, "total_obs_var"])
                model_var = float(_vc.loc[m,        "total_obs_var"])
                PVAR[m] = 100.0 * (1.0 - model_var / null_var)

# -- Load data ----------------------------------------------------------------
df    = pd.read_parquet("diagnostics/london_august_diagnostics.parquet")
df["ts"] = pd.to_datetime(df["hour"])
ranef = pd.read_csv("diagnostics/london_august_ranef.csv")
dwellings = df["dwelling"].unique()
print(f"{len(df):,} obs, {len(dwellings)} dwellings")

# -- Shared helpers -----------------------------------------------------------
def _label(ax, nm):
    rho, phi = AR_PARAMS[nm]
    ax.set_title(f"{nm} - {FAMILY[nm]}/{TODS[nm]}", fontsize=7, pad=3)
    if rho is not None:
        lines = [f"$\\rho$={rho:.4f}"]
        if phi is not None:
            lines.append(f"$\\Phi$={phi:.4f}")
        ax.text(0.97, 0.97, "\n".join(lines),
                transform=ax.transAxes, fontsize=6,
                va="top", ha="right")

def hide_unused(axes):
    for ax in axes.flat[len(MODELS):]:
        ax.set_visible(False)

MAX_LAG = 48
def mean_acf(col, max_lag=MAX_LAG):
    acfs = []
    for _, grp in df.groupby("dwelling", sort=False):
        grp = grp.sort_values("ts")
        r = grp[col].to_numpy()
        if len(r) < max_lag + 5:
            continue
        r = r - np.nanmean(r)
        if np.any(~np.isfinite(r)):
            continue
        v = np.var(r, ddof=0)
        if v < 1e-12:
            continue
        ac = np.array([
            np.mean(r[:len(r) - k] * r[k:]) / v
            for k in range(max_lag + 1)
        ])
        acfs.append(ac)
    if len(acfs) == 0:
        return np.full(max_lag + 1, np.nan)
    return np.mean(np.vstack(acfs), axis=0)

rng = np.random.default_rng(0)

## 1. AIC comparison

Marginal AIC (mAIC) and modelled-variance $R^2$ across the two model families.
AR(1) residuals are far better than i.i.d.; adding building characteristics
raises $R^2$ but barely changes mAIC.

In [ ]:
# load log-likelihoods (for AIC cross-check, not used in pvar)
_ll = {}
if os.path.exists(_aic_path):
    _aicdf = pd.read_csv(_aic_path).set_index("model")
    for _m in _aicdf.index:
        _ll[_m] = float(_aicdf.loc[_m, "loglik"])

from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(6.8, 2.6))
gs = GridSpec(1, 2, figure=fig, wspace=0.32)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

x       = np.arange(len(MODELS))
colours = [FAMILY_COLOUR[FAMILY[m]] for m in MODELS]

# Panel A: mAIC (log scale)
axA.bar(x, [AIC_VALUES[m] for m in MODELS],
        color=colours, edgecolor="black", linewidth=0.4, width=0.8)
axA.set_yscale("log")
axA.set_xticks(x)
axA.set_xticklabels(MODELS, rotation=35, ha="right", fontsize=7)
axA.tick_params(axis="y", labelsize=7)
axA.tick_params(axis="x", which="minor", bottom=False)
axA.set_ylabel("mAIC (log scale)", fontsize=8)
axA.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
axA.set_title("mAIC", fontsize=8)

# Panel B: modelled-variance R^2 vs family-specific null
if PVAR:
    pv = [PVAR.get(m, float("nan")) for m in MODELS]
    axB.bar(x, pv, color=colours, edgecolor="black", linewidth=0.4, width=0.8)
    axB.axhline(0, color="black", linewidth=0.6, linestyle="-")
    axB.set_xticks(x)
    axB.set_xticklabels(MODELS, rotation=35, ha="right", fontsize=7)
    axB.tick_params(axis="y", labelsize=7)
    axB.tick_params(axis="x", which="minor", bottom=False)
    axB.set_ylabel(r"Modelled variance $R^2$ (\%)", fontsize=8)
    axB.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
    axB.set_title(r"\% variance accounted for vs null", fontsize=8)

# Panel labels (match epc_tin_combined exactly)
for ax, lab in [(axA, "(A)"), (axB, "(B)")]:
    ax.text(-0.18, 1.06, lab, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="bottom", ha="left")

handles = [Patch(color=c, label=("i.i.d. residuals" if f == "Static" else "AR(1) residuals"))
           for f, c in FAMILY_COLOUR.items()]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.10),
           ncol=len(handles), fontsize=7, frameon=False)

plt.savefig(
    "plots/efus2017/london_livingroom_1month/mm_london_aic.svg",
    bbox_inches="tight",
)
plt.show()

## 2. Residual ACF: normalized residuals

ACF of nlme normalized residuals.

For **static** models: normalized residuals are standardized conditional residuals
and may still show serial autocorrelation.

For **AR(1)** models: normalized residuals are transformed by the fitted
correlation structure and should be approximately white noise if the residual
model is adequate. A residual peak at lag 24 indicates seasonal (diurnal)
autocorrelation that AR(1) does not capture.

Dashed lines are a rough per-dwelling white-noise reference: +/-1.96/sqrt(median
dwelling series length). Dotted vertical line = lag 24.

In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 8), sharex=True, sharey=True)
lags = np.arange(MAX_LAG + 1)

median_n = df.groupby("dwelling").size().median()
ci = 1.96 / np.sqrt(median_n)

for ax, nm in zip(axes.flat, MODELS):
    col = f"nresid_{nm}"
    if col not in df.columns:
        ax.text(
            0.5, 0.5,
            f"{col}\nnot found",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=8,
        )
        _label(ax, nm)
        continue
    acf = mean_acf(col)
    c = FAMILY_COLOUR[FAMILY[nm]]
    ax.bar(lags, acf, color=c, alpha=0.75, width=0.8)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axhline( ci, color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.axhline(-ci, color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.axvline(24, color="black", linewidth=0.5, linestyle=":", alpha=0.5)
    _label(ax, nm)
    ax.set_xlim(-0.5, MAX_LAG + 0.5)
    ax.set_ylim(-0.25, 1.0)
    ax.set_xlabel("Lag (hours)", fontsize=8)

hide_unused(axes)
for ax in axes[:, 0]:
    ax.set_ylabel("Mean ACF", fontsize=8)

fig.suptitle(
    "London August --- Mean ACF of nlme normalized residuals\n"
    r"Dashed $=$ per-dwelling white-noise reference ($\pm 1.96/\sqrt{n}$). Dotted $=$ lag 24 hrs.",
    fontsize=10,
)
fig.tight_layout()

fig.savefig(
    "plots/efus2017/london_livingroom_1month/mm_london_acf.svg",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


## 3. Normal QQ Plots (Normalised Residuals)

Tests the marginal distribution of the `nlme` normalised residuals. Deviations from the diagonal indicate heavy tails, skew, outliers, or remaining model misspecification after applying the fitted residual-correlation structure.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 10))

lims = [-5, 5]

for ax, nm in zip(axes.flat, MODELS):
    col = f"nresid_{nm}"

    r = (
        df[col].dropna().values
        if col in df.columns
        else df[f"resid_{nm}"].dropna().values
    )

    if len(r) > 5000:
        r = rng.choice(r, 5000, replace=False)

    (osm, osr), (slope, intercept, _) = stats.probplot(r, dist="norm")

    c = FAMILY_COLOUR[FAMILY[nm]]

    ax.scatter(
        osm, osr,
        s=3,
        alpha=0.4,
        color=c,
        rasterized=True,
    )

    x_line = np.array(lims)
    ax.plot(
        x_line,
        slope * x_line + intercept,
        color="black",
        linewidth=1,
    )

    _label(ax, nm)

    ax.set_xlabel("Theoretical quantiles", fontsize=7)
    ax.set_ylabel("Sample quantiles", fontsize=7)
    ax.tick_params(labelsize=7)

    # identical scales
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    # square panels
    ax.set_aspect("equal", adjustable="box")

hide_unused(axes)

fig.suptitle(
    "London August - Normal QQ of nlme normalised residuals",
    fontsize=11,
)

fig.tight_layout()

fig.savefig(
    "plots/efus2017/london_livingroom_1month/mm_london_qq.svg",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 4. Normalised Residuals vs Fitted Values

Checks for heteroskedasticity or nonlinear mean structure. Red = running mean (window=n/50). Should be flat at zero.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 9))
idx_sample = rng.choice(len(df), min(4000, len(df)), replace=False)

for ax, nm in zip(axes.flat, MODELS):
    fitted = df[f"fitted_{nm}"].values[idx_sample]
    col = f"nresid_{nm}"
    resid = (df[col].values[idx_sample] if col in df.columns
             else df[f"resid_{nm}"].values[idx_sample])
    c = FAMILY_COLOUR[FAMILY[nm]]
    ax.scatter(fitted, resid, s=3, alpha=0.3, color=c, rasterized=True)
    ax.axhline(0, color="black", linewidth=0.8)
    order = np.argsort(fitted)
    fs, rs = fitted[order], resid[order]
    w = max(1, len(fs) // 50)
    smooth = np.convolve(rs, np.ones(w) / w, mode="valid")
    ax.plot(fs[w//2: w//2 + len(smooth)], smooth, color="red", linewidth=1, alpha=0.8)
    _label(ax, nm)
    ax.set_xlabel("Fitted (deg C)", fontsize=7)
    ax.set_ylabel("Norm. resid.", fontsize=7)
    ax.tick_params(labelsize=7)

hide_unused(axes)
fig.suptitle("London August - nlme normalised residuals vs fitted (red = running mean, n=4000)",
             fontsize=11)
fig.tight_layout()
fig.savefig("plots/efus2017/london_livingroom_1month/mm_london_resid_fit.png", dpi=150)
plt.show()


## 5. Random Effects (BLUPs)

u₀ = dwelling-level intercept BLUP; u₁ = T_out slope BLUP. Correlation shown in top-left corner.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 9))

for ax, nm in zip(axes.flat, MODELS):
    if f"u0_{nm}" not in ranef.columns:
        ax.text(0.5, 0.5, f"u0_{nm}\nnot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=8)
        _label(ax, nm)
        continue
    u0 = ranef[f"u0_{nm}"].values
    u1 = ranef[f"u1_{nm}"].values
    c = FAMILY_COLOUR[FAMILY[nm]]
    ax.scatter(u0, u1, s=25, alpha=0.7, color=c, edgecolors="white", linewidths=0.3)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    if np.std(u0) > 1e-6 and np.std(u1) > 1e-6:
        corr = np.corrcoef(u0, u1)[0, 1]
        ax.text(0.05, 0.95, f"r = {corr:.3f}", transform=ax.transAxes,
                fontsize=7, va="top")
    _label(ax, nm)
    ax.set_xlabel("u0 intercept BLUP (deg C)", fontsize=7)
    ax.set_ylabel("u1 slope BLUP (deg C/deg C)", fontsize=7)
    ax.tick_params(labelsize=7)

hide_unused(axes)
fig.suptitle("London August - Random effects BLUPs (u0 = dwelling intercept, u1 = $T_{out}$ slope)",
             fontsize=11)
fig.tight_layout()
fig.savefig("plots/efus2017/london_livingroom_1month/mm_london_ranef.svg", dpi=150)
plt.show()


## 6. Time Series — One-Step-Ahead Predictions

Shows observed (black), marginal mean (dashed), and one-step-ahead (OSA) prediction (solid)
for one example dwelling (highest variance among median-length dwellings).

The marginal mean is flat relative to the diurnal cycle; the OSA prediction incorporates
the lagged AR state and tracks the slow August warming trend.


In [ ]:
# Choose example dwelling: median obs count, highest T_in variance
obs_counts = df.groupby("dwelling").size()
med = obs_counts.median()
candidates = obs_counts[abs(obs_counts - med) <= 10].index.tolist()
variances = df[df["dwelling"].isin(candidates)].groupby("dwelling")["T_in"].var()
example_dw = variances.idxmax()

dw_df = df[df["dwelling"] == example_dw].sort_values("ts").reset_index(drop=False)
n_show = min(len(dw_df), 21 * 24)
t = np.arange(n_show)

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(12, 8), sharex=True)

for ax, nm in zip(axes.flat, MODELS):
    c = FAMILY_COLOUR[FAMILY[nm]]
    rho, phi = AR_PARAMS[nm]
    fitted_vals = dw_df[f"fitted_{nm}"].values[:n_show]
    resid_vals  = dw_df[f"resid_{nm}"].values

    if rho is not None:
        osa = fitted_vals.copy()
        osa[1:] = fitted_vals[1:] + rho * resid_vals[:n_show - 1]
        if phi is not None:
            osa[24:] = osa[24:] + phi * resid_vals[:n_show - 24]
    else:
        osa = fitted_vals

    obs = dw_df["T_in"].values[:n_show]
    ax.plot(t, obs, color="black", linewidth=0.6, alpha=0.5, label="Observed", zorder=1)
    ax.plot(t, fitted_vals, color=c, linewidth=1.0, linestyle="--",
            alpha=0.6, label="Mean (fitted)", zorder=2)
    if rho is not None:
        ax.plot(t, osa, color=c, linewidth=1.2, label="OSA prediction", zorder=3)
    _label(ax, nm)
    ax.tick_params(labelsize=7)
    ax.set_ylabel("$T_{in}$ (deg C)", fontsize=7)

hide_unused(axes)
for ax in axes[NROWS - 1]:
    ax.set_xlabel("Hour index (first 21 days)", fontsize=8)

# Collect unique legend handles from all panels
handles, labels = [], []
for ax in axes.flat[:len(MODELS)]:
    for h, l in zip(*ax.get_legend_handles_labels()):
        if l not in labels:
            handles.append(h); labels.append(l)
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8,
           bbox_to_anchor=(0.5, -0.01))

fig.suptitle(
    f"London August - Dwelling {example_dw}: observed (black), marginal mean (dashed),\n"
    "one-step-ahead prediction (solid, AR/SAR models only).",
    fontsize=10)
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("plots/efus2017/london_livingroom_1month/mm_london_timeseries.svg",
            dpi=150, bbox_inches="tight")
plt.show()


## 7. Variance Components and Residual-Correlation Parameters

Residual standard deviation scale sigma_epsilon, AR coefficient rho, and seasonal AR coefficient Phi (lag 24). In these `nlme` models, sigma_epsilon is the residual variance scale used with the fitted within-group correlation structure; it should not be labelled as marginal residual standard deviation or as a standalone marginal residual standard deviation in the comparison table.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
x = np.arange(len(MODELS))
colours = [FAMILY_COLOUR[FAMILY[m]] for m in MODELS]

sig_u0 = [SIG_U0[m] for m in MODELS]
axes[0].bar(x, sig_u0, color=colours, edgecolor="white", linewidth=0.5)
axes[0].set_xticks(x); axes[0].set_xticklabels(MODELS, fontsize=8, rotation=45, ha="right")
axes[0].set_title(r"Random-intercept SD $\sigma_{u0}$ (deg C)", fontsize=9)
axes[0].set_ylabel(r"$\sigma_{u0}$ (deg C)")

sig_eps = [SIG_EPS[m] for m in MODELS]
axes[1].bar(x, sig_eps, color=colours, edgecolor="white", linewidth=0.5)
axes[1].set_xticks(x); axes[1].set_xticklabels(MODELS, fontsize=8, rotation=45, ha="right")
axes[1].set_title(r"Residual SD scale $\sigma_{\varepsilon}$ (deg C)", fontsize=9)
axes[1].set_ylabel(r"$\sigma_{\varepsilon}$ (deg C)")

rho_vals = [AR_PARAMS[m][0] if AR_PARAMS[m][0] is not None else 0 for m in MODELS]
axes[2].bar(x, rho_vals, color=colours, edgecolor="white", linewidth=0.5)
axes[2].set_xticks(x); axes[2].set_xticklabels(MODELS, fontsize=8, rotation=45, ha="right")
axes[2].set_title(r"AR coefficient $\rho$", fontsize=9)
axes[2].set_ylabel(r"$\rho$")

handles = [Patch(color=c, label=("i.i.d. residuals" if f == "Static" else "AR(1) residuals"))
           for f, c in FAMILY_COLOUR.items()]
axes[0].legend(handles=handles, fontsize=8)
fig.suptitle("London August --- Variance components and AR coefficient (M1-M14)", fontsize=10)
fig.tight_layout()
fig.savefig("plots/efus2017/london_livingroom_1month/mm_london_variance_components.png", dpi=150)
plt.show()

## 8. Variance Accounted For by Fixed Model

Percentage of observation-level variance accounted for by the fixed part of each model, calculated as the reduction in total observation-level variance relative to the corresponding constant-only null model from the same residual-correlation family:

$$
\% \text{ variance accounted for}
= 100 \times \left(1 - \frac{V_{\text{model}}}{V_{\text{null}}}\right)
$$

where:

- `M0_static` is the constant-only null for the static models;
- `M0_ar1` is the constant-only null for the AR(1) models;
- `M0_sar24` is the constant-only null for the SAR(1,24) models.

The variance quantity used here is `total_obs_var` exported from R, not the simple sum `sig_u0^2 + sig_u1^2 + sig_eps^2`. For these models,

$$
V_{\text{model}}
= \operatorname{E}\left[\operatorname{Var}(u_0 + u_1 T_{out,c})\right]
+ \sigma^2_\varepsilon,
$$

so the calculation includes the random intercept variance, random slope variance, their covariance, the observed distribution of `T_out_c`, and the residual variance scale from `nlme`.

The fitted null model changes by family so that models are compared only with a null model having the same random-effects and residual-correlation structure. Negative values are retained because they indicate that the fitted model has a larger estimated total observation-level variance than its family-specific null.


In [ ]:
# ── Model specification + variance components + modelled-variance R^2 ─────────
# Emits a LaTeX table in the tab:models style, extended with the random/residual
# variance components and the total explained variance (R^2 vs family null).
_vc_tbl = pd.read_csv("diagnostics/london_august_varcomp.csv").set_index("model")

required_cols = {"sig_u0", "sig_u1", "sig_eps", "cov_u01", "total_obs_var"}
missing_cols = required_cols.difference(_vc_tbl.columns)
if missing_cols:
    raise ValueError(
        "diagnostics/london_august_varcomp.csv is missing required columns: "
        + ", ".join(sorted(missing_cols)) + ". Re-run the updated R script."
    )

NULL_FOR_TBL = {"Static": "M0_static", "AR(1)": "M0_ar1", "SAR(1,24)": "M0_sar24"}
FAMILY_LABEL = {"Static": "i.i.d.", "AR(1)": "AR(1)", "SAR(1,24)": "SAR(1,24)"}
TOD_LABEL    = {"none": "None", "sin/cos": "Sine/cosine", "hour": "Hour dummies"}


def split_tod_bldg(t):
    """Split the notebook's ToD code into (time-of-day term, building chars)."""
    base = t.split("+")[0]
    tod = TOD_LABEL[base]
    if "cat" in t:
        bldg = "Binary + categorical"
    elif "bin" in t:
        bldg = "Binary"
    else:
        bldg = "None"
    return tod, bldg


rows = []
for m in MODELS:
    if m not in _vc_tbl.index:
        continue
    v = _vc_tbl.loc[m]
    tod, bldg = split_tod_bldg(TODS[m])
    null_key = NULL_FOR_TBL[FAMILY[m]]
    null_var = float(_vc_tbl.loc[null_key, "total_obs_var"])
    r2 = 100.0 * (1.0 - float(v["total_obs_var"]) / null_var)
    rows.append({
        "Model": m,
        "Family": FAMILY_LABEL[FAMILY[m]],
        "ToD": tod,
        "Building": bldg,
        "s2_u0": float(v["sig_u0"]) ** 2,
        "s2_u1": float(v["sig_u1"]) ** 2,
        "s2_eps": float(v["sig_eps"]) ** 2,
        "mAIC": float(AIC_VALUES[m]),
        "R2": r2,
    })

tbl = pd.DataFrame(rows).set_index("Model")
display(tbl.style.format({
    "s2_u0": "{:.3f}", "s2_u1": "{:.4f}", "s2_eps": "{:.3f}",
    "mAIC": "{:.1f}", "R2": "{:.1f}%",
}))

# ── LaTeX (tab:models style) ──────────────────────────────────────────────────
header = [
    r"\begin{table}[ht]",
    r"  \centering",
    r"  \small",
    r"  \begin{tabular*}{\textwidth}{@{\extracolsep{\fill}} l r r r r r @{}}",
    r"    \toprule",
    r"    Model & $\sigma^2_{u0}$ & $\sigma^2_{u1}$ & $\sigma^2_{\varepsilon}$ "
    r"& mAIC & $R^2$ (\%) \\",
    r"    \midrule",
]
body = []
for i, (m, r) in enumerate(tbl.iterrows()):
    if m == "M8":                       # split i.i.d. block from AR(1) block
        body.append(r"    \midrule")
    body.append(
        f"    {m} & {r['s2_u0']:.3f} & {r['s2_u1']:.4f} & "
        f"{r['s2_eps']:.3f} & {r['mAIC']:.1f} & {r['R2']:.1f} \\\\"
    )
footer = [
    r"    \bottomrule",
    r"  \end{tabular*}",
    r"  \caption{Variance components and modelled-variance $R^2$ for the model "
    r"specifications in Table~\ref{tab:models}. $\sigma^2_{u0}$, $\sigma^2_{u1}$ and "
    r"$\sigma^2_{\varepsilon}$ are the random-intercept, random-slope and residual "
    r"variances ($^\circ$C$^2$); $R^2$ is the proportional reduction in the "
    r"observation-level (random + residual) variance relative to the "
    r"family-specific intercept-only null model.}",
    r"  \label{tab:model_variance}",
    r"\end{table}",
]
print("\n".join(header + body + footer))